# FLAIR on Kaggle — GPU runs

This notebook runs **only the parts that need a GPU**: SD3.5-M generation and,
later, the BASM calibration campaign.

Everything else — parsing, fuzzy hedges, routing decisions, the whole test
suite — runs locally with no GPU. Don't burn quota on it:

```bash
docker build -t flair-test .
docker run --rm flair-test                                    # 87 tests
docker run --rm flair-test python scripts/explain.py "a very red car"
```

## Before you start

1. **Settings → Accelerator → GPU T4 ×2** (or P100)
2. **Accept the SD3.5-Medium licence** on its Hugging Face model page — it is a
   *gated* repo and the download 403s without this. One time only.
3. **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face read token)

## What you get

Two numbers this notebook prints — `N_BLOCKS` and `T_GEN` — set the entire
calibration budget. Write them down (roadmap §2.1).

## 1 · Authenticate

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded")

## 2 · Clone and install

If the repo is private, replace the URL with
`https://<user>:<github-token>@github.com/sunzidiautomation/FD.git`.

In [ ]:
!git clone -q https://github.com/sunzidiautomation/FD.git /kaggle/working/FD
%cd /kaggle/working/FD
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

## ⚠️ 3 · Restart the session now

**Run → Restart session**, then continue from cell 4. Do not skip this.

Freshly-installed packages are not visible to the already-running kernel.
Skipping the restart is exactly what produced the `ModuleNotFoundError: No
module named 'lpips'` and `'skfuzzy'` errors in the earlier notebook — the
install cell had run fine; the kernel just could not see it.

## 4 · Verify the environment

Must print `Environment OK.` before going further.

In [ ]:
%cd /kaggle/working/FD
!python scripts/verify_env.py

## 5 · Sanity check without the GPU

Cheap confirmation that the clone is intact before pulling 5GB of weights.

In [ ]:
!python -m pytest -q
!python scripts/explain.py "A small red sports car under warm evening light" --steps 8

## 6 · Smoke test — the first real images

Downloads SD3.5-M (~5GB) on first run, then generates five images.
Expect roughly 10–20 minutes total.

In [ ]:
!python scripts/smoke_test.py --steps 20 --seed 0 --out /kaggle/working/outputs

## 7 · Look at the results

### How to judge these — read before you do

The BASM is **uncalibrated**. `BASM.uniform()` sets every cell to 0.5, and
`top_k` breaks ties by ascending block id, so **every attribute routes to
block 0**. All four streams pile into one block.

This run proves the *plumbing*, not that routing helps. Judge it on:

- ✅ five images exist, none is noise
- ✅ baseline and routed differ from each other
- ✅ the log shows 4 routed components and `blocks touched: [0]`
- ❌ **not** on colour or size fidelity — nothing is calibrated to route by yet

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

out = Path("/kaggle/working/outputs")
names = [
    "smoke_baseline",
    "smoke_routed",
    "smoke_hedge_slightly",
    "smoke_hedge_plain",
    "smoke_hedge_very",
]

found = [(n, out / f"{n}.png") for n in names if (out / f"{n}.png").exists()]
if not found:
    raise SystemExit(f"no images in {out} -- did cell 6 run?")

fig, axes = plt.subplots(1, len(found), figsize=(4 * len(found), 4.5))
axes = axes if len(found) > 1 else [axes]
for ax, (name, path) in zip(axes, found):
    ax.imshow(Image.open(path))
    ax.set_title(name.replace("smoke_", ""), fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8 · Record the two measurements

Copy `N_BLOCKS` and `T_GEN` from cell 6's output into the cell below and run
it. **The calibration campaign's parameters are derived from these** — see the
master roadmap §2.2:

```
total generations = A × P × S × (1 + B)
P × S  ≤  (18 × 3600) / (7 × (1 + B) × T_GEN)
```

where A = 7 attributes, P = pairs per attribute, S = seeds, B = vital blocks.

In [ ]:
N_BLOCKS = 24    # <-- from cell 6
T_GEN = 12.4     # <-- from cell 6, seconds

from pathlib import Path

runs = Path("/kaggle/working/FD/calibration_runs")
runs.mkdir(parents=True, exist_ok=True)
(runs / "measurements.txt").write_text(
    f"N_BLOCKS={N_BLOCKS}\nT_GEN={T_GEN}\n"
)

A, B, WEEKLY_GPU_HOURS = 7, 10, 18
budget = (WEEKLY_GPU_HOURS * 3600) / (A * (1 + B) * T_GEN)
print(f"prefilter cost   ~{3 * (1 + N_BLOCKS)} generations")
print(f"P x S budget     <= {budget:.1f}   (at B={B})")
print(f"  P=5,  S=1  ->  {A * 5 * 1 * (1 + B)} generations")
print(f"  P=10, S=1  ->  {A * 10 * 1 * (1 + B)} generations")
print(f"  P=10, S=2  ->  {A * 10 * 2 * (1 + B)} generations   <- spec target")

## 9 · Calibration — not yet available

The cell below needs **Tasks 11-16** (metrics, corpus, prefilter, harness),
which are planned in `docs/superpowers/plans/…-calibration-harness.md` but not
built. It will fail until they are.

**Do Task 16b first** — `calibrate()` currently has no checkpointing, and a
1500-generation sweep can outlast Kaggle's 12-hour session cap and lose
everything (roadmap §2.4).

Choose `--top-n` from the vitality elbow: keep blocks down to where the score
falls below 50% of the top block's, clamped to [6, 12].

In [ ]:
# Phase 1 -- vital-layer prefilter (cheap, ~75 generations)
# !python scripts/calibrate.py prefilter --top-n 10 --out calibration_runs/

# Phase 2 -- the BASM sweep (the expensive one; resumable after Task 16b)
# !python scripts/calibrate.py basm \
#     --vitality calibration_runs/vitality.json \
#     --seeds 0 1 --out calibration_runs/

## 10 · Save your work

Kaggle wipes everything outside `/kaggle/working` when the session ends, and
even that is lost unless you **Save Version**. `basm.npz` is the expensive
artefact — it costs hours of GPU to regenerate. Download it, and commit it to
the repo once calibration succeeds.

In [ ]:
!ls -la /kaggle/working/outputs/ 2>/dev/null
!ls -la /kaggle/working/FD/calibration_runs/ 2>/dev/null